In [ ]:
# Baseline — KNN / SVM / RF Comparison (7 Iterative Improvements)
# Progression: 1-subject → 5-subject expansion → feature engineering
#              → undersampling (failed) → class_weight → RF final
# Final: KNN 62.69% / SVM 64.93% / RF 67.91%

import mne
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from collections import Counter
import matplotlib.pyplot as plt
from mne.datasets.sleep_physionet.age import fetch_data

stage_names = {
    'Wake': 'Sleep stage W',
    'N1':   'Sleep stage 1',
    'N2':   'Sleep stage 2',
    'N3':   'Sleep stage 3',
    'REM':  'Sleep stage R',
}

def extract_features(signal, fs):
    fft_result = np.fft.rfft(signal)
    powers     = np.abs(fft_result) ** 2
    freqs      = np.fft.rfftfreq(len(signal), d=1/fs)
    delta = np.sum(powers[(freqs >= 0.5) & (freqs < 4)])
    theta = np.sum(powers[(freqs >= 4)   & (freqs < 8)])
    alpha = np.sum(powers[(freqs >= 8)   & (freqs < 13)])
    beta  = np.sum(powers[(freqs >= 13)  & (freqs < 30)])
    theta_beta  = theta / (beta  + 1e-6)   # 1e-6: 0 나누기 방지 / REM vs Wake 구분
    delta_theta = delta / (theta + 1e-6)   # N3 vs N1 구분
    return [delta, theta, alpha, beta, theta_beta, delta_theta]

all_X = []
all_y = []

for subject_id in [0, 1, 2, 3, 4]:
    files    = fetch_data(subjects=[subject_id], recording=[1])
    psg_file = files[0][0]
    hyp_file = files[0][1]
    raw         = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)
    fs_real     = int(raw.info['sfreq'])
    data, _     = raw['EEG Fpz-Cz']
    annotations = mne.read_annotations(hyp_file)

    for stage_label, stage_raw in stage_names.items():
        for j in range(len(annotations)):
            if annotations.description[j] == stage_raw:
                onset     = annotations.onset[j]
                start_idx = int(onset * fs_real)
                end_idx   = int((onset + 30) * fs_real)
                if end_idx <= data.shape[1]:
                    epoch    = data[0, start_idx:end_idx]
                    features = extract_features(epoch, fs_real)
                    all_X.append(features)
                    all_y.append(stage_label)

X_real = np.array(all_X)
y_real = np.array(all_y)

X_train, X_test, y_train, y_test = train_test_split(
    X_real, y_real, test_size=0.2, random_state=42,
    stratify=y_real 
)

scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grid Search - RF
rf_params = {
    'n_estimators': [100, 200],
    'max_depth':    [None, 10, 20],
}
rf_grid = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42),
    rf_params, cv=cv, scoring='accuracy'
)
rf_grid.fit(X_train_scaled, y_train)

y_pred_rf_best = rf_grid.predict(X_test_scaled)

# Feature Importance
feature_names = ['Delta', 'Theta', 'Alpha', 'Beta', 'Theta/Beta', 'Delta/Theta']
importances   = rf_grid.best_estimator_.feature_importances_

plt.figure(figsize=(8, 4))
plt.barh(feature_names, importances)
plt.xlabel('Importance')
plt.title('Feature Importance (Random Forest)')
plt.tight_layout()
plt.show()

# Grid Search - KNN / SVM 
knn_params = {'n_neighbors': [1, 3, 5, 7, 9, 11]}
knn_grid   = GridSearchCV(KNeighborsClassifier(), knn_params, cv=cv, scoring='accuracy')
knn_grid.fit(X_train_scaled, y_train)
y_pred_knn_best = knn_grid.predict(X_test_scaled)

svm_params = {'C': [0.1, 1, 10], 'gamma': [0.01, 0.1, 1]}
svm_grid = GridSearchCV(
    SVC(kernel='rbf', class_weight='balanced', random_state=42),
    svm_params, cv=cv, scoring='accuracy'
)
svm_grid.fit(X_train_scaled, y_train)
y_pred_svm_best = svm_grid.predict(X_test_scaled)

print("\n===== Final Model Performance Comparison =====")
print(f"RF Test Accuracy:{accuracy_score(y_test, y_pred_rf_best):.2%}")
print(f"KNN Test Accuracy:{accuracy_score(y_test, y_pred_knn_best):.2%}")
print(f"SVM Test Accuracy:{accuracy_score(y_test, y_pred_svm_best):.2%}")

# Confusion Matrix Comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, y_pred, title in zip(
    axes,
    [y_pred_knn_best, y_pred_svm_best, y_pred_rf_best],
    ['KNN', 'SVM', 'Random Forest']
):
    cm = confusion_matrix(y_test, y_pred, labels=list(stage_names.keys()))
    ConfusionMatrixDisplay(cm, display_labels=list(stage_names.keys())).plot(
        cmap='Blues', ax=ax
    )
    ax.set_title(f"{title} — Acc: {accuracy_score(y_test, y_pred):.2%}")

plt.suptitle("Confusion Matrix Comparison — KNN / SVM / RF", fontsize=13)
plt.tight_layout()
plt.savefig("baseline_confusion_matrix_comparison.png", dpi=150, bbox_inches='tight')
plt.show()